In [1]:
import pennylane as qml
import numpy as np
from math import log2, isclose

In [2]:
def state_to_ket(state, *, tol=1e-10, max_terms=None, binary=True, normalize_check=False):
    """
    Convert a statevector to Dirac ket notation.

    Parameters
    ----------
    state : array-like of complex
        1D array of amplitudes, length = 2^n (typically).
    tol : float
        Amplitudes with |amp| <= tol are omitted.
    max_terms : int or None
        If set, shows only the largest `max_terms` amplitudes by magnitude.
    binary : bool
        If True, basis labels are |00...1>. If False, labels are |k>.
    normalize_check : bool
        If True, raises ValueError if state is not ~normalized.

    Returns
    -------
    str : a Dirac-notation string
    """
    v = np.asarray(state, dtype=complex).flatten()
    if v.ndim != 1:
        raise ValueError("state must be a 1D array-like.")
    dim = v.size
    if dim == 0:
        return "0"

    # Optional normalization check
    if normalize_check:
        norm = float(np.vdot(v, v).real)
        if not isclose(norm, 1.0, rel_tol=1e-9, abs_tol=1e-9):
            raise ValueError(f"State not normalized: ||ψ||^2 = {norm}")

    # Determine qubit count if possible
    n = None
    if dim & (dim - 1) == 0:  # power of two
        n = int(log2(dim))

    # Collect nonzero terms
    terms = [(i, amp) for i, amp in enumerate(v) if abs(amp) > tol]
    if not terms:
        return "0"

    # Optionally keep only largest amplitudes
    if max_terms is not None and max_terms > 0 and len(terms) > max_terms:
        terms.sort(key=lambda x: abs(x[1]), reverse=True)
        terms = terms[:max_terms]
        terms.sort(key=lambda x: x[0])

    def fmt_complex(z):
        # Nicely format complex numbers with small real/imag suppressed.
        r = 0.0 if abs(z.real) < tol else z.real
        im = 0.0 if abs(z.imag) < tol else z.imag
        if im == 0.0:
            return f"{r:.6g}"
        if r == 0.0:
            return f"{im:.6g}j"
        sign = "+" if im >= 0 else "-"
        return f"({r:.6g}{sign}{abs(im):.6g}j)"

    def basis_label(i):
        if binary:
            if n is None:
                return f"|{i}⟩"
            return f"|{format(i, f'0{n}b')}⟩"
        return f"|{i}⟩"

    pieces = []
    for i, amp in terms:
        label = basis_label(i)

        # Handle special cases for amplitude formatting
        if abs(amp - 1) <= tol:
            coeff = ""
            sign = "+"
        elif abs(amp + 1) <= tol:
            coeff = ""
            sign = "-"
        else:
            coeff = fmt_complex(amp)
            sign = "+"

        pieces.append((sign, coeff, label))

    # Build string with proper leading sign
    out = []
    for idx, (sign, coeff, label) in enumerate(pieces):
        if idx == 0:
            if sign == "-":
                out.append("-")
            # if coeff empty, just label; else coeff + label
            out.append(f"{coeff}{label}" if coeff else f"{label}")
        else:
            out.append(f" {sign} ")
            out.append(f"{coeff}{label}" if coeff else f"{label}")

    return "".join(out)


def print_state_ket(state, **kwargs):
    """Convenience: print the ket string."""
    print(state_to_ket(state, **kwargs))


In [3]:
bra0 = np.array([[1,0]])
bra1 = np.array([[0,1]])
ket0 = np.transpose(bra0)
ket1 = np.transpose(bra1)
id = np.array([[1,0],[0,1]])
x = np.array([[0,1],[1,0]])
cx = np.kron(np.kron(ket0,bra0),id) + np.kron(np.kron(ket1,bra1),x)

In [4]:
ket00 = np.kron(ket0,ket0)
bra00 = np.transpose(np.conjugate(ket00))
ket11 = np.kron(ket1,ket1)
bra11 = np.transpose(np.conjugate(ket11))

In [5]:
tof = np.kron(np.kron(ket0,bra0),np.kron(id,id)) + np.kron(np.kron(ket1,bra1),cx)
print(tof)

[[1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 1 0]]


In [6]:
t1=np.kron(np.kron(ket0,bra1),np.kron(ket0,bra1))
t2=np.kron(ket00,bra11)
print(t1==t2)

[[ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]
 [ True  True  True  True]]


In [7]:
dev = qml.device("default.qubit", 3)

@qml.qnode(dev)
def test_tof(qbits,state):
    qml.StatePrep(state, wires=range(3), normalize=True) 
    qml.Toffoli(qbits)
    return qml.state()

In [8]:
st0 = np.array([1,0])
st1 = np.array([0,1])
states = [
    np.kron(np.kron(i, j), k)
    for i in (st0, st1)
    for j in (st0, st1)
    for k in (st0, st1)
]

for st in states:
    print(state_to_ket(st), ' -> ', state_to_ket(test_tof([0,2,1],st)))

|000⟩  ->  |000⟩
|001⟩  ->  |001⟩
|010⟩  ->  |010⟩
|011⟩  ->  |011⟩
|100⟩  ->  |100⟩
|101⟩  ->  |111⟩
|110⟩  ->  |110⟩
|111⟩  ->  |101⟩


In [35]:
def prepState(nwires):
    states_int = [n for n in range(2**nwires)]
    states_bin = [np.binary_repr(n,nwires) for n in states_int]
    return states_bin

def qcirc():
    qml.CNOT([0,1])
    qml.CNOT([0,2])
    qml.CNOT([2,0])

@qml.qnode(dev)
def appCirc(qcirc,state,nwires):
    qml.BasisState([int(b) for b in state], wires=range(nwires))
    qcirc()
    return qml.state()
    

for st in prepState(3):
    st_f = appCirc(qcirc, st, 3)
    print(st, '->', state_to_ket(st_f))

000 -> |000⟩
001 -> |101⟩
010 -> |010⟩
011 -> |111⟩
100 -> |011⟩
101 -> |110⟩
110 -> |001⟩
111 -> |100⟩
